# 02. Прогнозирование уровня тревожности

Регрессионные модели для предсказания GAD-7 оценки.

**Модели:** RandomForest, LightGBM, CatBoost
**Тюнинг:** Optuna
**Логирование:** MLflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [ ]:
df = pd.read_csv('data/processed_data.csv')

TARGET = 'Anxiety_Level'
FEATURES = [c for c in df.columns if c not in [TARGET, 'Subjective_Anxiety']]

X = df[FEATURES]
y = df[TARGET]

print(f"Features: {len(FEATURES)}, Samples: {len(y)}")
print(f"Target range: {y.min()} - {y.max()}")

## Разбиение данных

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=pd.qcut(y, q=5, labels=False, duplicates='drop')
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Вспомогательные функции

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, name=''):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    n, p = len(y_test), X_test.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    
    print(f"\n{name}")
    print(f"  MAE: {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  R²: {r2:.3f}")
    print(f"  Adj R²: {adj_r2:.3f}")
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'adj_r2': adj_r2, 'predictions': y_pred}

## Baseline: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_baseline = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_results = evaluate_model(rf_baseline, X_train, X_test, y_train, y_test, 'Random Forest Baseline')

## Optuna: Random Forest

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestRegressor(**params)
    
    stratifier = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_strat = pd.qcut(y_train, q=5, labels=False, duplicates='drop')
    
    scores = []
    for train_idx, val_idx in stratifier.split(X_train, y_strat):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(mean_absolute_error(y_val, pred))
    
    return np.mean(scores)

study_rf = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=True)

print(f"\nЛучший MAE: {study_rf.best_value:.3f}")
print(f"Параметры: {study_rf.best_params}")

In [ ]:
rf_optuna = RandomForestRegressor(**study_rf.best_params, random_state=42, n_jobs=-1)
rf_opt_results = evaluate_model(rf_optuna, X_train, X_test, y_train, y_test, 'Random Forest + Optuna')

## LightGBM

In [ ]:
import lightgbm as lgb

def objective_lgb(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 8, 64),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42
    }
    
    model = lgb.LGBMRegressor(**params)
    
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    return -np.mean(scores)

study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

print(f"\nЛучший MAE: {study_lgb.best_value:.3f}")
print(f"Параметры: {study_lgb.best_params}")

In [ ]:
lgb_model = lgb.LGBMRegressor(**study_lgb.best_params, verbosity=-1, random_state=42)
lgb_results = evaluate_model(lgb_model, X_train, X_test, y_train, y_test, 'LightGBM + Optuna')

## CatBoost

In [ ]:
from catboost import CatBoostRegressor

def objective_cat(trial):
    params = {
        'loss_function': 'MAE',
        'iterations': trial.suggest_int('iterations', 50, 300),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'random_seed': 42,
        'verbose': 0
    }
    
    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    return -np.mean(scores)

study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=30, show_progress_bar=True)

print(f"\nЛучший MAE: {study_cat.best_value:.3f}")

In [ ]:
cat_model = CatBoostRegressor(**study_cat.best_params, random_seed=42, verbose=0)
cat_results = evaluate_model(cat_model, X_train, X_test, y_train, y_test, 'CatBoost + Optuna')

## Сравнение моделей

In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'RF + Optuna', 'LightGBM', 'CatBoost'],
    'MAE': [rf_results['mae'], rf_opt_results['mae'], lgb_results['mae'], cat_results['mae']],
    'RMSE': [rf_results['rmse'], rf_opt_results['rmse'], lgb_results['rmse'], cat_results['rmse']],
    'R²': [rf_results['r2'], rf_opt_results['r2'], lgb_results['r2'], cat_results['r2']]
})
results = results.sort_values('MAE')
print(results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(results['Model'], results['MAE'], color='steelblue')
axes[0].set_title('MAE (меньше = лучше)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(results['Model'], results['RMSE'], color='coral')
axes[1].set_title('RMSE (меньше = лучше)')
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(results['Model'], results['R²'], color='seagreen')
axes[2].set_title('R² (больше = лучше)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Feature Importance (LightGBM)

In [ ]:
lgb_model.fit(X_train, y_train)
importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='mediumpurple')
plt.xlabel('Importance')
plt.title('Feature Importance (LightGBM)')
plt.tight_layout()
plt.show()

## Сохранение лучшей модели

In [ ]:
import joblib

best_model = lgb_model  
joblib.dump(best_model, 'models/best_model.pkl')
print("Модель сохранена: models/best_model.pkl")